# Term-by-term check of the quadratic effective Spinfoam action

This notebook computes the numerical ingredients used in the quadratic effective Spinfoam action. It follows the order used in the manuscript: construct the Lorentzian geometry, evaluate the critical action, compute the derivative and Hessian data, and then compare the analytic formulas term by term.

The final section gives the main checks. The boundary linear term is compared with the first Regge variation. The Regge quadratic term is split into its bulk and boundary parts. The non-Regge correction is built from the correction matrix $\mathcal M$. The direct stationary-phase result is then compared with the sum of the Regge quadratic term and the non-Regge correction.


In [1]:
cd(joinpath(pwd(), "src/LorentzianSimplexSolver"))

using Pkg
Pkg.activate(".")
Pkg.instantiate()
using LorentzianSimplexSolver

  Activating project at `~/Documents/Work/effective-spinfoam/code/Effective-Spinfoam/src/LorentzianSimplexSolver`


In [2]:
include("../../scripts/run_geometry.jl");
include("../../scripts/run_action.jl");
include("../../scripts/run_dlogEh_dX.jl")
include("../../scripts/run_deta_dl.jl")
include("../perturbations/TransverseBasis.jl")
include("../perturbations/Soln_dY_dX.jl")
include("../perturbations/DθDl.jl");

In [3]:
using .RunGeometry
using .RunAction
using .RunDlogEhDX
using .DηDLUtils
using .TransverseBasis
using .Soln_dY_dX

## 1. Geometry and input data

The following cells define the triangulation, vertex coordinates, precision, Immirzi parameter, and the vertex-coordinate dictionary used by the geometry solver.


In [ ]:
simplices = [[1,2,3,4,6], [1,2,3,5,6], [1,2,4,5,6],[1,2,3,4,7], [1,2,3,5,7], [1,2,4,5,7]]

all_vertices = unique(Iterators.flatten(simplices))
sort!(all_vertices)

coords_lines = [
    "0, 0, 0, 0",
    "-0.068000000000000005, -0.21988127663727278, -0.5316227766016838, -1.3316227766016839",
    "0, 0, 0, -3.398088489694245",
    "-0.24028114141347542, -0.6936319083813028, -0.9809436521275706, -1.6990442448471226",
    "0, 0, -2.942830956382712, -1.6990442448471226",
    "0, -2.7745276335252114, -0.9809436521275706, -1.6990442448471226",
    "-2.4696884592430974, -3.893218630529324, -1.3565336794679874, -1.9090667752920147",
]

coords_lines = [
    "0, 0, 0, 0",
    "0.3, 0, 0, 1",
    "0.4, 0, 1, 0",
    "0.7, 0, 1, 1",
    "0.5, 1, 0, 0",
    "0.8, 1, 0, 1",
    "0.9, 1, 1, 0",
    "1.2, 1, 1, 1",
    "1.6, 1.5, 1.25, 1",
    "1.9, 1.5, 1.25, 2",
    "2, 1.5, 2.25, 1",
    "2.3, 1.5, 2.25, 2",
    "2.1, 2.5, 1.25, 1",
    "2.4, 2.5, 1.25, 2",
    "2.5, 2.5, 2.25, 1",
    "2.8, 2.5, 2.25, 2",
]

const ScalarT = Float64
# tol = 1e-10;
# const ScalarT = BigFloat
const tol = parse(ScalarT, "1e-8")

if ScalarT === BigFloat
    setprecision(BigFloat, 80)
    LorentzianSimplexSolver.PrecisionUtils.set_big_precision!(80)
    # LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(sqrt(eps(BigFloat)))
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(tol)
else
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(tol)
end

gamma_vals = ScalarT(1.0);

vertex_coords = Dict{Int, Vector{ScalarT}}()  

for (v, line) in zip(all_vertices, coords_lines)
    vertex_coords[v] = LorentzianSimplexSolver.PrecisionUtils.parse_numeric_line(line, ScalarT)
end

In [5]:
geom = run_geometry_pipeline(simplices, coords_lines, ScalarT, tol);

## 2. Critical action and variables

Here we reconstruct the Lorentzian geometry, compute the Regge data, and evaluate the Spinfoam action at the critical point. These objects fix the background used in the later expansion.


In [6]:
deficit_angles, dihedral_angles, _, _, iRegge = LorentzianSimplexSolver.ReggeAction.run_Regge_action(geom, simplices, vertex_coords);

In [7]:
γ = LorentzianSimplexSolver.DefineAction.γsym()
sd, S_symbols, phase_soln = RunAction.run_action(geom, dihedral_angles, γ);

g_vars = geom.varias[:g_var]
z_vars = geom.varias[:z_var]
η_vars = geom.varias[:η_var]

vars = vcat(g_vars, z_vars, η_vars);

In [8]:
using SymEngine
vals = LorentzianSimplexSolver.ActionEvaluation.build_value_dict(sd, γ; γval=gamma_vals);

S = LorentzianSimplexSolver.ActionEvaluation.eval_symbolic(S_symbols, phase_soln);
S_val = LorentzianSimplexSolver.ActionEvaluation.eval_symbolic(S, vals);
SF_action = SymEngine.expand(S_val);


## 3. Bulk derivative data

This block computes the derivatives of the bulk face factors with respect to the Spinfoam variables $X^\alpha=(g,z)$. These derivatives enter the Hessian reduction and the correction matrix.


In [9]:
nh = length(geom.connectivity[1]["OrderBulkFaces"])

dlogEh_dX_sym = RunDlogEhDX.run_dlogEh_dX(geom)

dlogEh_dX_vals = RunDlogEhDX.evaluate_dlogEh_dX(dlogEh_dX_sym, geom, sd; γval=gamma_vals);

## 4. Boundary derivative data

The boundary expansion uses derivatives with respect to both $X^\alpha$ and the boundary variables $Y$. These are the numerical ingredients for the linear and boundary quadratic terms.


In [10]:
nb = length(geom.connectivity[1]["OrderBDryFaces"])

dlogEb_dX_sym, dlogEb_dY_sym, Y_vars = RunDlogEhDX.run_dlogEb_dXY(geom);

dlogEb_dX_vals, dlogEb_dY_vals = RunDlogEhDX.evaluate_dlogEb_dXY(dlogEb_dX_sym, dlogEb_dY_sym, Y_vars, geom, sd, phase_soln; γval=gamma_vals);

## 5. Boundary Taylor coefficients

These cells compute the derivatives of the boundary contribution $\sum_b k_b\log E_b$. They are used below to form the boundary linear term and the quadratic boundary contribution.


In [11]:
dkbEb_dX_sym, dkbEb_dY_sym, d2kbEb_dXdY_sym, d2kbEb_dYdY_sym = RunDlogEhDX.run_kblogEb_dXY(geom, Y_vars);

In [12]:
dkbEb_dX_vals, dkbEb_dY_vals, d2kbEb_dXdY_vals, d2kbEb_dYdY_vals = RunDlogEhDX.evaluate_kblogEb_all(dkbEb_dX_sym, dkbEb_dY_sym, d2kbEb_dXdY_sym, d2kbEb_dYdY_sym, geom, sd, Y_vars, phase_soln; γval=gamma_vals);

In [13]:
d2kbEb_dXdY_vals_sumb = sum(d2kbEb_dXdY_vals[i, :, :] for i in 1:nb);

## 6. Regge directions in the bulk face variables

The matrix $\partial\eta_h/\partial\ell_s$ describes the part of the bulk face variation generated by edge-length variations. The transverse basis computed next spans the non-Regge directions.


In [13]:
# dηdl_matrix is a nh x nl matrix
η_h_vertices = DηDLUtils.get_bulk_faces_vertices(geom)

bulk_edges, bdry_edges = DηDLUtils.get_bulk_edges(geom, η_h_vertices)
bdry_edges_perturb = [bdry_edges[1]] # here we only perturb one boundary edge
perturb_edges = vcat(bulk_edges, bdry_edges_perturb)
    
dηdl_matrix = DηDLUtils.build_dηdl_matrix(η_h_vertices, perturb_edges, vertex_coords, ScalarT, gamma_vals);
nl = length(perturb_edges)
nt = nh - nl;

## 7. Transverse non-Regge directions

The columns of `eListHT` give a numerical basis for the directions in bulk face space that are not generated by length variations.


In [15]:
# eListHT is a nh x nt matrix
eListHT = TransverseBasis.compute_transverse_basis(dηdl_matrix, tol);

## 8. Boundary edge perturbation data

The boundary spins are varied through the chosen Regge-like boundary length perturbation. These first and second derivatives enter the boundary part of the Regge expansion.


In [16]:
dkbdl, d2kb_dldl = Soln_dY_dX.dkb_dl(geom, nl, bdry_edges_perturb, vertex_coords ;γ=gamma_vals);

## 9. Linearized bulk and boundary angles

The matrices `DϵDl` and `DΘDl` are the numerical versions of the linearized bulk deficit angles and boundary dihedral angles.


In [17]:
DϵDl = DθDl_module.compute_dθDl(simplices, η_h_vertices, perturb_edges, vertex_coords, geom.connectivity[1]["Tets"], ScalarT);

In [18]:
bd_faces = geom.connectivity[1]["OrderBDryFaces"];
kb_vertices = [geom.connectivity[1]["TetFaces"][f[1][1]][f[1][2]][f[1][3]] for f in bd_faces];
DΘDl = DθDl_module.compute_dθDl(simplices, kb_vertices, perturb_edges, vertex_coords, geom.connectivity[1]["Tets"], ScalarT);

## 10. Hessian of the Spinfoam action

This block computes the Hessian with respect to the Spinfoam variables. It is the matrix used in the stationary-phase reduction.


In [19]:
H_symbols = LorentzianSimplexSolver.EOMsHessian.compute_Hessian_block_half(S, vars);
H_eval = LorentzianSimplexSolver.EOMsHessian.evaluate_hessian_block(H_symbols, sd; γ=gamma_vals);

## 11. Linear response of the critical variables

The following matrices describe how the critical Spinfoam variables respond to the chosen length perturbation. They are the numerical form of the linearized equations used in the manuscript.


In [20]:
ng = length(g_vars)
nz = length(z_vars)
nX = ng + nz
X_vars = vcat(g_vars, z_vars)

invHessianXX = inv(H_eval[1:nX, 1:nX]);

In [21]:
Bα = transpose(transpose(dηdl_matrix) * dlogEh_dX_vals);

In [22]:
M_matrix = vcat(
    dlogEb_dY_vals[:, nb+1:end] -
        dlogEb_dX_vals * invHessianXX * d2kbEb_dXdY_vals_sumb[:, nb+1:end],
    -dlogEh_dX_vals * invHessianXX * d2kbEb_dXdY_vals_sumb[:, nb+1:end],
);

dmatrix = vcat(
    im * gamma_vals/2 * DΘDl +
        dlogEb_dX_vals * invHessianXX *
        (Bα + d2kbEb_dXdY_vals_sumb[:, 1:nb] * dkbdl),
    im * gamma_vals/2 * DϵDl +
        dlogEh_dX_vals * invHessianXX *
        (Bα + d2kbEb_dXdY_vals_sumb[:, 1:nb] * dkbdl),
);

using LinearAlgebra
DξDl = pinv(M_matrix, atol=tol, rtol=tol) * dmatrix;


In [23]:
DYDl = vcat(dkbdl, DξDl);

In [24]:
DXDl = -invHessianXX * (Bα + d2kbEb_dXdY_vals_sumb[:, 1:nb] * dkbdl + d2kbEb_dXdY_vals_sumb[:, nb+1:end] * DξDl);

In [25]:
Cα =d2kbEb_dXdY_vals_sumb * DYDl;

## 12. Second variation of the boundary variables

The second variation of $Y$ gives the terms in the boundary expansion that contain the second boundary length variation.


In [26]:
kb_vals = [ScalarT(subs(Y_vars[i], vals)) for i in 1:nb];

Mm = [
    kb_vals[b] * dlogEb_dY_vals[b, j]
    for b in 1:nb, j in nb+1:length(Y_vars)
];

R = [
    dkbdl[i, j] * transpose(dlogEb_dY_vals[i, nb+1:end]) * DξDl[:, j] +
    transpose(DXDl[:, j]) * d2kbEb_dXdY_vals[i, :, nb+1:end] * DξDl[:, j] +
    transpose(DξDl[:, j]) * d2kbEb_dYdY_vals[i, nb+1:end, nb+1:end] * DξDl[:, j]
    for i in 1:nb, j in 1:nl
];

D2ξDl2 = -pinv(Mm, atol=tol, rtol=tol) * R[:, end];


In [27]:
nxi = length(Y_vars) - nb;

D2ξDl2_matrix = [
    j == k == nl ? D2ξDl2[b] : 0
    for b in 1:nxi, j in 1:nl, k in 1:nl
];

D2YDl2_matrix = vcat(d2kb_dldl, D2ξDl2_matrix);


## 13. Boundary contribution

This part computes the boundary linear term and the boundary contribution to the quadratic expansion. In the manuscript these terms are compared with the boundary part of the Regge expansion.


In [28]:
BA = vcat(Bα + Cα, zeros(nt, nl));


In [29]:
eta_h = [
    LorentzianSimplexSolver.DefineSymbols.make_symbol("η_$(faces[1][1])$(faces[1][2])$(faces[1][3])")
    for faces in geom.connectivity[1]["OrderBulkFaces"]
];

eta_h_vals = [ScalarT(subs(eta_h[i], vals)) for i in 1:nh];
Ahh = Matrix(Diagonal(eta_h_vals));

hαβ = H_eval[1:nX, 1:nX] + transpose(dlogEh_dX_vals) * Ahh * dlogEh_dX_vals;
Hαi = transpose(dlogEh_dX_vals) * eListHT;

HIJ = [hαβ Hαi; transpose(Hαi) zeros(nt, nt)];
invHIJ = inv(HIJ);


In [30]:
DtXDl = -invHIJ * BA;

In [31]:
d2kbEb_dYdY_vals_sumb = sum(d2kbEb_dYdY_vals[i, :, :] for i in 1:nb);
dkbEb_dY_vals_sumb = sum(dkbEb_dY_vals[i, :] for i in 1:nb);

Iboundary_linear = transpose(dkbEb_dY_vals_sumb) * DYDl;


In [32]:
Iboundary_quadratic =
    1/2 * transpose(DYDl) * d2kbEb_dYdY_vals_sumb * DYDl +
    1/2 * sum(
        dkbEb_dY_vals_sumb[i] * D2YDl2_matrix[i, :, :]
        for i in 1:length(Y_vars)
    );

# Keep the old variable name for compatibility with earlier cells.
Iboundary_qadratic = Iboundary_quadratic;


## 14. Direct stationary-phase quadratic term

The matrix `SF_quadratic_form2` is the direct numerical stationary-phase result. It is computed before rewriting the answer in terms of the Regge quadratic action and the non-Regge correction.


In [33]:
SF_quadratic =
    transpose(BA) * DtXDl +
    1/2 * transpose(DtXDl) * HIJ * DtXDl +
    Iboundary_quadratic;


In [34]:
SF_quadratic_form2 =
    -1/2 * transpose(BA[1:nX, :]) * invHIJ[1:nX, 1:nX] * BA[1:nX, :] +
    Iboundary_quadratic;


In [35]:
SF_linear_bdry = Iboundary_linear;


## 15. Regge terms and the non-Regge correction matrix

Here we build the matrices entering the correction formula in the notation of the manuscript. The derivative matrix of the bulk face factors is

$$
V_{\alpha h}=\left[\partial_\alpha\log E_h\right]_0,
\qquad
A_{hh'}=\bar\eta_h\delta_{hh'} .
$$

The Woodbury matrix is

$$
w=-V^T H^{-1}V,
\qquad
\rho=(A^{-1}-w)^{-1} .
$$

The transverse part is written with the basis $\hat e^i_h$ and the Schur complement $S$,

$$
\kappa=\hat e S^{-1}\hat e^T,
\qquad
\kappa_{hh'}=\sum_{i,j}\hat e_{hi}(S^{-1})^{ij}\hat e_{h'j} .
$$

The correction matrix stored in the code as `M_kernel` is

$$
\mathcal M_{hh'}
=
\rho_{hh'}
-
\bigl(\rho A^{-1}\kappa A^{-1}\rho\bigr)_{hh'} .
$$

The corresponding non-Regge correction is

$$
\Delta S^{(2)}
=
-
\frac{\gamma^2}{8}
\sum_{h,h'}
\mathcal M_{hh'}\,
\delta\varepsilon_h\delta\varepsilon_{h'} .
$$

Here $\delta\varepsilon_h=\varepsilon^{(1)}_{h,r}\delta\ell_r$. In the code, the matrix of coefficients $\varepsilon^{(1)}_{h,r}$ is stored as `DϵDl`, and the quadratic matrix after substituting $\delta\varepsilon_h$ is stored as `correction_term`.


In [36]:
ω = -dlogEh_dX_vals * invHessianXX * transpose(dlogEh_dX_vals);
ρ = inv(inv(Ahh) - ω);


In [37]:
Sij =
    -transpose(eListHT) * dlogEh_dX_vals * inv(hαβ) *
    transpose(dlogEh_dX_vals) * eListHT;

κ = eListHT * inv(Sij) * transpose(eListHT);

M_kernel = ρ - ρ * inv(Ahh) * κ * inv(Ahh) * ρ;


In [38]:
correction_term = -gamma_vals^2/8 * transpose(DϵDl) * M_kernel * DϵDl;


In [39]:
iSRegge_quadratic = im * (
    gamma_vals/4 * transpose(dηdl_matrix) * DϵDl +
    gamma_vals/4 * (
        transpose(dkbdl) * DΘDl +
        sum(dihedral_angles[i] * d2kb_dldl[i, :, :] for i in 1:nb)
    )
);


In [40]:
SF_quadratic_wrt_deficit = iSRegge_quadratic + correction_term;


In [41]:
iSRegge_linear = im/2 * gamma_vals * transpose(dkbdl) * dihedral_angles;


## 16. Paper-style term-by-term comparison

The cells below only rename and group the quantities already computed above. This makes the numerical check match the structure of the manuscript.

The comparison checks

$$
\delta \mathcal I_{\rm bdry}= iS_{\rm Regge}^{(1)},
\qquad
(S'_0-\bar S'_0)^{(2)}_{\rm direct}
= iS_{\rm Regge}^{(2)}+\Delta S^{(2)} .
$$

We also display the split of $iS_{\rm Regge}^{(2)}$ into bulk and boundary parts before doing the final comparison.


In [42]:
using LinearAlgebra

function print_term_summary(name, value; show_matrix=true)
    println(name)
    println("  size = ", size(value))
    println("  sum  = ", sum(value))
    println("  norm = ", norm(value))
    if show_matrix
        show(stdout, "text/plain", value)
        println()
    end
    return value
end

function report_comparison(name, lhs, rhs)
    diff = lhs - rhs
    println(name)
    println("  size(lhs)       = ", size(lhs))
    println("  size(rhs)       = ", size(rhs))
    println("  norm(lhs-rhs)   = ", norm(diff))
    println("  max |lhs-rhs|   = ", maximum(abs.(diff)))
    println("  sum(lhs-rhs)    = ", sum(diff))
    return diff
end


report_comparison (generic function with 1 method)

### 16.1 Linear term

The direct Spinfoam boundary expansion gives `SF_linear_bdry`. The Regge first variation gives `iSRegge_linear`. The check below verifies

$$
\delta \mathcal I_{\rm bdry}
=
\frac{i\gamma}{2}
\sum_b k^{(1)}_{b,r}\bar\Theta_b\,\delta\ell_r
= iS_{\rm Regge}^{(1)} .
$$


In [43]:
spinfoam_linear_term = transpose(SF_linear_bdry)
regge_linear_term = iSRegge_linear

print_term_summary("Regge linear term iS_Regge^(1)", regge_linear_term)
println()

linear_difference = report_comparison(
    "Linear term: Spinfoam boundary expansion = Regge first variation",
    spinfoam_linear_term,
    regge_linear_term,
)


Regge linear term iS_Regge^(1)
  size = (2,)
  sum  = 0.00000000000000000 - 2.4781224413627930*im
  norm = 2.478122441362792964841459544234991058184598770241281069226720383020251679088025
2-element Vector{Any}:
                                           0
 0.00000000000000000 - 2.4781224413627930*im

Linear term: Spinfoam boundary expansion = Regge first variation
  size(lhs)       = (2,)
  size(rhs)       = (2,)
  norm(lhs-rhs)   = 9.643517057537659121934710150173963256159164461142269869674837513458958781253882e-12
  max |lhs-rhs|   = 7.3216911366624548e-12
  sum(lhs-rhs)    = 8.6844555669773025e-12 - 5.9231924564035461e-12*im


2-element Vector{Basic}:
     1.36556839856817e-12 - 6.12580468895069e-12*im
 7.3188871684091300e-12 + 2.0261223254713912e-13*im

### 16.2 Regge quadratic term

For the comparison we first display the Regge quadratic term as a sum of its bulk and boundary pieces,

$$
iS_{\rm Regge}^{(2)}
=iS_{{\rm Regge},{\rm bulk}}^{(2)}
+iS_{{\rm Regge},{\rm bdry}}^{(2)} .
$$

Using the notation of the manuscript,

$$
iS_{{\rm Regge},{\rm bulk}}^{(2)}
=
\frac{i\gamma}{4}
\sum_h
\eta^{(1)}_{h,r}\varepsilon^{(1)}_{h,s}
\delta\ell_r\delta\ell_s,
$$

and

$$
iS_{{\rm Regge},{\rm bdry}}^{(2)}
=
\frac{i\gamma}{4}
\sum_b
\left[
 k^{(1)}_{b,r}\Theta^{(1)}_{b,s}
 +k^{(2)}_{b,rs}\bar\Theta_b
\right]
\delta\ell_r\delta\ell_s .
$$

The next cell checks that these two pieces reproduce the full `iSRegge_quadratic` matrix.


In [44]:
iSRegge_bulk_quadratic = im * gamma_vals/4 * transpose(dηdl_matrix) * DϵDl

iSRegge_boundary_quadratic = im * gamma_vals/4 * (
    transpose(dkbdl) * DΘDl +
    sum(dihedral_angles[i] * d2kb_dldl[i, :, :] for i in 1:nb)
)

print_term_summary("Regge quadratic bulk term iS_Regge,bulk^(2)", iSRegge_bulk_quadratic)
println()
print_term_summary("Regge quadratic boundary term iS_Regge,bdry^(2)", iSRegge_boundary_quadratic)
println()
print_term_summary("Full Regge quadratic term iS_Regge^(2)", iSRegge_quadratic)
println()

regge_quadratic_split_difference = report_comparison(
    "Regge quadratic split: bulk + boundary = full Regge quadratic term",
    iSRegge_bulk_quadratic + iSRegge_boundary_quadratic,
    iSRegge_quadratic,
)


Regge quadratic bulk term iS_Regge,bulk^(2)
  size = (2, 2)
  sum  = 0.0 - 9.04638782008329im
  norm = 212.40195834449244
2×2 Matrix{ComplexF64}:
 -0.0-134.089im   0.0+152.539im
  0.0+28.0163im  -0.0-55.5126im

Regge quadratic boundary term iS_Regge,bdry^(2)
  size = (2, 2)
  sum  = 0.00000000000000000 + 62.282447671461882*im
  norm = 139.2112135499638938328836505434585373677756880458374389522033779624987099971728
2×2 Matrix{Basic}:
                                0.0 + 0.0*im                                  0.0 + 0.0*im
 0.00000000000000000 + 124.52272718109227*im  -0.00000000000000000 - 62.240279509630385*im

Full Regge quadratic term iS_Regge^(2)
  size = (2, 2)
  sum  = 0.00000000000000000 + 53.236059851378596*im
  norm = 279.9677109999730354707775161404186316817777996203497427788799962718152577999519
2×2 Matrix{Basic}:
                   -0.0 - 134.08908832298*im                     0.0 + 152.539007806566*im
 0.00000000000000000 + 152.53900780656630*im  -0.00000000000000000 - 117

2×2 Matrix{Basic}:
 0  0
 0  0

### 16.3 Non-Regge correction

The matrix `M_kernel` is the correction matrix $\mathcal M_{hh'}$. The correction term is

$$
\Delta S^{(2)}
=
-
\frac{\gamma^2}{8}
\sum_{h,h'}
\mathcal M_{hh'}\,
\delta\varepsilon_h\delta\varepsilon_{h'} .
$$

The code stores the corresponding quadratic matrix in the length perturbations as `correction_term`.


In [45]:
println("correction matrix M size = ", size(M_kernel))
println("correction term size     = ", size(correction_term))
println("norm(correction term)    = ", norm(correction_term))
println("sum(correction term)     = ", sum(correction_term))

correction matrix M size = (5, 5)
correction term size     = (2, 2)
norm(correction term)    = 277.0819445645681
sum(correction term)     = -25.280603133320167 - 33.20247529459736im


### 16.4 Final equality

The final check compares the direct stationary-phase quadratic result with the expression used in the paper. Combining the linear and quadratic pieces gives

$$
S'_0-\bar S'_0
= iS_{\rm Regge}^{(1)}
+iS_{\rm Regge}^{(2)}
-
\frac{\gamma^2}{8}
\sum_{h,h'}
\mathcal M_{hh'}\,
\delta\varepsilon_h\delta\varepsilon_{h'}
+O(3).
$$

Since the linear term was checked in Sec. 16.1, the code below compares the quadratic matrices.


In [46]:
direct_spinfoam_quadratic = SF_quadratic_form2
regge_plus_correction_quadratic = iSRegge_quadratic + correction_term

quadratic_difference = report_comparison(
    "Quadratic term: direct Spinfoam stationary phase = Regge quadratic + non-Regge correction",
    direct_spinfoam_quadratic,
    regge_plus_correction_quadratic,
)

comparison_summary = [
    (; term = "linear", norm = norm(linear_difference), max_abs = maximum(abs.(linear_difference)), total = sum(linear_difference)),
    (; term = "Regge quadratic split", norm = norm(regge_quadratic_split_difference), max_abs = maximum(abs.(regge_quadratic_split_difference)), total = sum(regge_quadratic_split_difference)),
    (; term = "quadratic final", norm = norm(quadratic_difference), max_abs = maximum(abs.(quadratic_difference)), total = sum(quadratic_difference)),
]

Quadratic term: direct Spinfoam stationary phase = Regge quadratic + non-Regge correction
  size(lhs)       = (2, 2)
  size(rhs)       = (2, 2)
  norm(lhs-rhs)   = 1.199083293493282056218185378530834921202727995369298842873696609269104256309867e-11
  max |lhs-rhs|   = 7.5087661886629699e-12
  sum(lhs-rhs)    = 8.5317586329125561e-12 + 6.2751193130594629e-12*im


3-element Vector{NamedTuple{(:term, :norm, :max_abs, :total)}}:
 (term = "linear", norm = 9.643517057537659121934710150173963256159164461142269869674837513458958781253882e-12, max_abs = 7.3216911366624548e-12, total = 8.6844555669773025e-12 - 5.9231924564035461e-12*im)
 (term = "Regge quadratic split", norm = 0.0, max_abs = 0, total = 0)
 (term = "quadratic final", norm = 1.199083293493282056218185378530834921202727995369298842873696609269104256309867e-11, max_abs = 7.5087661886629699e-12, total = 8.5317586329125561e-12 + 6.2751193130594629e-12*im)